# Runtime: numethods vs SciPy

A pure-Python implementation will not beat SciPy on speed—SciPy calls into LAPACK and other compiled routines. The point of this notebook is to **document the gap honestly** and show that on small-to-medium problems the difference is modest, while on large problems SciPy pulls ahead by orders of magnitude.

Each row times the same operation on the same input. We use `timeit` with a small repeat count.


In [1]:
import time
import numpy as np
import scipy.integrate as spi
import scipy.linalg as spl
import scipy.optimize as spo

import numethods as nm

rng = np.random.default_rng(0)

def time_it(fn, repeats=5):
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best


## Linear systems: solve a 200x200 dense system

In [2]:
n = 200
A = rng.standard_normal((n, n)) + n * np.eye(n)
b = rng.standard_normal(n)

t_us = time_it(lambda: nm.gauss_elimination(A, b))
t_sp = time_it(lambda: spl.solve(A, b))

print(f"numethods.gauss_elimination : {t_us*1e3:7.2f} ms")
print(f"scipy.linalg.solve          : {t_sp*1e3:7.2f} ms")
print(f"speed gap                   : {t_us/t_sp:.1f}x")


numethods.gauss_elimination :    6.03 ms
scipy.linalg.solve          :    0.23 ms
speed gap                   : 26.8x


## Quadrature: $\int_0^\pi \sin(x)\,dx$

In [3]:
t_us = time_it(lambda: nm.simpson(np.sin, 0, np.pi, 1000))
t_sp = time_it(lambda: spi.quad(np.sin, 0, np.pi))
print(f"numethods.simpson(n=1000)   : {t_us*1e6:7.1f} us")
print(f"scipy.integrate.quad        : {t_sp*1e6:7.1f} us")


numethods.simpson(n=1000)   :    19.9 us
scipy.integrate.quad        :     9.9 us


## ODE integration: stiff-ish exponential decay

In [4]:
def f(t, y): return -y
t_us = time_it(lambda: nm.rk4(f, (0, 10.0), 1.0, n_steps=1000))
t_sp = time_it(lambda: spi.solve_ivp(f, (0, 10.0), [1.0], method="RK45", rtol=1e-8))
print(f"numethods.rk4 (1000 steps)  : {t_us*1e3:7.2f} ms")
print(f"scipy.solve_ivp (RK45)      : {t_sp*1e3:7.2f} ms")


numethods.rk4 (1000 steps)  :    6.45 ms
scipy.solve_ivp (RK45)      :    0.59 ms


## Root-finding: $\cos(x) - x = 0$

In [5]:
f = lambda x: np.cos(x) - x
fp = lambda x: -np.sin(x) - 1
t_us = time_it(lambda: nm.newton(f, 0.5, fp))
t_sp = time_it(lambda: spo.newton(f, 0.5, fprime=fp))
print(f"numethods.newton           : {t_us*1e6:7.1f} us")
print(f"scipy.optimize.newton      : {t_sp*1e6:7.1f} us")


numethods.newton           :     7.6 us
scipy.optimize.newton      :    56.5 us


## Summary

| Operation                    | Expected gap | Why |
|------------------------------|--------------|-----|
| Linear solve (LU)            | 10–100x      | LAPACK in compiled Fortran |
| Quadrature (small n)         | comparable   | Both vectorized |
| ODE (fixed-step RK4)         | 2–10x        | scipy.solve_ivp has Python overhead |
| Scalar Newton                | comparable   | Both pure Python loops |

**Bottom line:** `numethods` is fast enough for teaching, exploration, and small-scale numerical work. For production at scale, use SciPy.